# DenseNet-121 Production-Aligned GLCM Fusion Comparison

Run this notebook after the production-ROI robustness experiment. It performs a paired, validation-only comparison of the selected DenseNet-121 control against an additive six-feature GLCM branch over three seeds. The CNN Grad-CAM explains only the CNN logit contribution; the notebook reports the GLCM contribution separately.

The notebook never opens the repeatedly inspected test split. It saves checkpoints, predictions, patient-clustered bootstrap intervals, full histories, classification reports, confusion matrices, CAM audits, example overlays, an acceptance decision, and `report.md` under a timestamped Google Drive directory.

Before running, review the path constants in the first code cell. By default, `BASE_CHECKPOINT = None` selects the newest completed checkpoint under `densenet121_production_roi_robustness`.


In [ ]:
%pip -q install "timm>=1.0" "ultralytics>=8.3,<9" "scikit-image>=0.23" "pandas>=2" "tabulate>=0.9"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.0 MB/s eta 0:00:00


In [ ]:
import gc
import json
import math
import random
import shutil
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from skimage.feature import graycomatrix, graycoprops
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm
from ultralytics import YOLO

try:
    from google.colab import drive

    drive.mount('/content/drive')
except ImportError:
    print('Google Colab is not active; using the configured local paths.')


# Update Drive paths only when your layout differs. By default the notebook
# selects the newest completed production-ROI robustness checkpoint.
INPUT_SIZE = 384
BATCH_SIZE = 48
NUM_WORKERS = 2
EPOCHS = 5
BACKBONE_LR = 1e-5
GLCM_BRANCH_LR = 1e-4
WEIGHT_DECAY = 1e-3
SEEDS = (42, 1337, 2026)
YOLO_CONFIDENCE = 0.45
YOLO_IMAGE_SIZE = 640
PRODUCTION_EXPANSION = 1.15
TRAIN_EXPANSION_RANGE = (1.10, 1.20)
TRAIN_SHIFT_FRACTION = 0.05
CAM_CASES_PER_GRADE = 20
CAM_EXAMPLES_PER_GRADE = 3
BOOTSTRAP_RESAMPLES = 1000

TEXTURE_HEIGHT = 104
TEXTURE_WIDTH = 224
GLCM_LEVELS = 32
GLCM_DISTANCES = (1, 2, 3)
GLCM_ANGLES = (0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4)
GLCM_PROPERTIES = (
    'contrast',
    'dissimilarity',
    'homogeneity',
    'energy',
    'correlation',
    'ASM',
)

DATASET_ROOT = Path('/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1')
PUBLISHED_ROOT = DATASET_ROOT / 'extracted/KneeXrayData/ClsKLData/kneeKL224'
FULL_PNG_ROOT = DATASET_ROOT / 'derived/full_bilateral_png_v2'
YOLO_CHECKPOINT = Path('/content/drive/MyDrive/Models/yolov8_checkpoint/best.pt')
MANIFEST_PATH = DATASET_ROOT / 'derived/production_yolo_roi_manifest_v1.csv'
GLCM_CACHE_PATH = DATASET_ROOT / 'derived/production_yolo_glcm_q32_104x224_v1.csv'
BASE_RUN_ROOT = Path('/content/drive/MyDrive/Models/densenet121_production_roi_robustness')
BASE_CHECKPOINT = None  # Set an explicit Path here to pin a particular run.
OUTPUT_ROOT = Path('/content/drive/MyDrive/Models/densenet121_glcm_fusion')
NOTEBOOK_SOURCE = Path('/content/dense_net_121_glcm_fusion_comparison.ipynb')

if BASE_CHECKPOINT is None:
    candidates = list(BASE_RUN_ROOT.glob('*/best_model.pth'))
    if not candidates:
        raise FileNotFoundError(
            'No production-ROI robustness checkpoint found. Run that experiment '
            'first or set BASE_CHECKPOINT explicitly.'
        )
    BASE_CHECKPOINT = max(candidates, key=lambda path: path.stat().st_mtime)

for required in (PUBLISHED_ROOT, FULL_PNG_ROOT, YOLO_CHECKPOINT, BASE_CHECKPOINT):
    if not required.exists():
        raise FileNotFoundError(f'Required path not found: {required}')

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_%f_UTC')
RUN_DIR = OUTPUT_ROOT / RUN_TIMESTAMP
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('Device candidate:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Base checkpoint:', BASE_CHECKPOINT)
print('Run directory:', RUN_DIR)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
seed_everything(SEEDS[0])



Mounted at /content/drive
Device candidate: cuda
Base checkpoint: /content/drive/MyDrive/Models/densenet121_production_roi_robustness/2026-07-30_15-45-03_830205_UTC/best_model.pth
Run directory: /content/drive/MyDrive/Models/densenet121_glcm_fusion/2026-07-31_00-49-00_429979_UTC


In [ ]:
def published_labels():
    labels = {}
    for split in ('train', 'val'):
        for grade in range(5):
            for path in (PUBLISHED_ROOT / split / str(grade)).glob('*.png'):
                side = path.stem[-1].upper()
                patient = path.stem[:-1]
                if side not in ('R', 'L'):
                    raise RuntimeError(f'Cannot infer knee side from {path.name}')
                labels[(split, patient, side)] = grade
    return labels


def build_or_load_manifest():
    if MANIFEST_PATH.is_file():
        frame = pd.read_csv(
            MANIFEST_PATH,
            dtype={'split': str, 'patient': str, 'side': str},
        )
    else:
        labels = published_labels()
        detector = YOLO(str(YOLO_CHECKPOINT))
        detector_device = 0 if torch.cuda.is_available() else 'cpu'
        rows = []
        for split in ('train', 'val'):
            paths = sorted((FULL_PNG_ROOT / split).glob('*.png'))
            for start in range(0, len(paths), BATCH_SIZE):
                batch_paths = paths[start : start + BATCH_SIZE]
                images = [cv2.imread(str(path), cv2.IMREAD_COLOR) for path in batch_paths]
                if any(image is None for image in images):
                    raise RuntimeError(f'Cannot decode a full image in {split}')
                predictions = detector.predict(
                    source=images,
                    conf=YOLO_CONFIDENCE,
                    imgsz=YOLO_IMAGE_SIZE,
                    device=detector_device,
                    batch=BATCH_SIZE,
                    save=False,
                    verbose=False,
                )
                for path, prediction in zip(batch_paths, predictions):
                    boxes = prediction.boxes.xyxy.detach().cpu().numpy()
                    scores = prediction.boxes.conf.detach().cpu().numpy()
                    boxes = boxes[np.argsort(scores)[::-1][:2]]
                    boxes = sorted(boxes, key=lambda box: float(box[0] + box[2]))
                    if len(boxes) != 2:
                        raise RuntimeError(
                            f'Expected two YOLO knees for {path}; found {len(boxes)}'
                        )
                    for box, side in zip(boxes, ('R', 'L')):
                        grade = labels.get((split, path.stem, side))
                        if grade is None:
                            raise RuntimeError(
                                f'Missing KL label for {split}/{path.stem}{side}'
                            )
                        rows.append(
                            {
                                'split': split,
                                'patient': path.stem,
                                'side': side,
                                'grade': grade,
                                'full_image': str(path),
                                'x1': float(box[0]),
                                'y1': float(box[1]),
                                'x2': float(box[2]),
                                'y2': float(box[3]),
                            }
                        )
                print(f'{split}: {min(start + len(batch_paths), len(paths))}/{len(paths)}')
        frame = pd.DataFrame(rows)
        MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
        frame.to_csv(MANIFEST_PATH, index=False)

    required_columns = {
        'split',
        'patient',
        'side',
        'grade',
        'full_image',
        'x1',
        'y1',
        'x2',
        'y2',
    }
    if not required_columns.issubset(frame.columns):
        raise RuntimeError(f'Invalid manifest: {MANIFEST_PATH}')
    if frame.duplicated(['split', 'patient', 'side']).any():
        raise RuntimeError('Manifest has duplicate split/patient/side rows')

    train_patients = set(frame.loc[frame.split == 'train', 'patient'])
    val_patients = set(frame.loc[frame.split == 'val', 'patient'])
    overlap = train_patients & val_patients
    if overlap:
        raise RuntimeError(
            f'Train/validation are not patient-disjoint; overlap={len(overlap)}'
        )
    return frame


def make_square_roi(image, box, expansion, shift_fraction=0.0):
    height, width = image.shape[:2]
    x1, y1, x2, y2 = map(float, box)
    box_width, box_height = x2 - x1, y2 - y1
    if box_width <= 0 or box_height <= 0:
        raise ValueError(f'Invalid YOLO box: {box}')
    side = int(math.ceil(max(box_width, box_height) * expansion))
    center_x, center_y = (x1 + x2) / 2, (y1 + y2) / 2
    center_x += random.uniform(-shift_fraction, shift_fraction) * side
    center_y += random.uniform(-shift_fraction, shift_fraction) * side
    wanted_x1 = int(math.floor(center_x - side / 2))
    wanted_y1 = int(math.floor(center_y - side / 2))
    wanted_x2, wanted_y2 = wanted_x1 + side, wanted_y1 + side
    crop = image[
        max(0, wanted_y1) : min(height, wanted_y2),
        max(0, wanted_x1) : min(width, wanted_x2),
    ]
    if crop.size == 0:
        raise RuntimeError(f'ROI has no overlap with the source image: {box}')
    return cv2.copyMakeBorder(
        crop,
        max(0, -wanted_y1),
        max(0, wanted_y2 - height),
        max(0, -wanted_x1),
        max(0, wanted_x2 - width),
        cv2.BORDER_CONSTANT,
        value=(0, 0, 0),
    )


def apply_clahe_rgb(image_rgb):
    lab = cv2.cvtColor(np.asarray(image_rgb), cv2.COLOR_RGB2LAB)
    lightness, channel_a, channel_b = cv2.split(lab)
    lightness = cv2.createCLAHE(
        clipLimit=1.25,
        tileGridSize=(8, 8),
    ).apply(lightness)
    return cv2.cvtColor(
        cv2.merge((lightness, channel_a, channel_b)),
        cv2.COLOR_LAB2RGB,
    )


class OpenCVCLAHE:
    def __call__(self, image_rgb):
        return apply_clahe_rgb(image_rgb)


class SquarePad:
    def __call__(self, image_rgb):
        image = np.asarray(image_rgb)
        height, width = image.shape[:2]
        side = max(height, width)
        top, left = (side - height) // 2, (side - width) // 2
        return cv2.copyMakeBorder(
            image,
            top,
            side - height - top,
            left,
            side - width - left,
            cv2.BORDER_CONSTANT,
            value=(0, 0, 0),
        )


manifest = build_or_load_manifest()
print(manifest.groupby(['split', 'grade']).size().unstack(fill_value=0))
print('Patient-disjoint manifest:', MANIFEST_PATH)



grade     0     1     2    3    4
split                            
train  2286  1046  1516  757  173
val     328   153   212  106   27
Patient-disjoint manifest: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/production_yolo_roi_manifest_v1.csv


In [ ]:
def extract_glcm_features(image_rgb):
    enhanced = apply_clahe_rgb(image_rgb)
    grayscale = cv2.cvtColor(enhanced, cv2.COLOR_RGB2GRAY)
    texture = cv2.resize(
        grayscale,
        (TEXTURE_WIDTH, TEXTURE_HEIGHT),
        interpolation=cv2.INTER_AREA,
    )
    quantized = np.minimum(
        (texture.astype(np.uint16) * GLCM_LEVELS) // 256,
        GLCM_LEVELS - 1,
    ).astype(np.uint8)
    matrix = graycomatrix(
        quantized,
        distances=list(GLCM_DISTANCES),
        angles=list(GLCM_ANGLES),
        levels=GLCM_LEVELS,
        symmetric=True,
        normed=True,
    )
    values = []
    for property_name in GLCM_PROPERTIES:
        value = float(np.nanmean(graycoprops(matrix, property_name)))
        values.append(value if np.isfinite(value) else 0.0)
    return np.asarray(values, dtype=np.float32)


def build_or_load_glcm_cache(frame):
    if GLCM_CACHE_PATH.is_file():
        cache = pd.read_csv(
            GLCM_CACHE_PATH,
            dtype={'split': str, 'patient': str, 'side': str},
        )
    else:
        rows = []
        for row in tqdm(frame.itertuples(index=False), total=len(frame), desc='GLCM'):
            image = cv2.imread(row.full_image, cv2.IMREAD_COLOR)
            if image is None:
                raise RuntimeError(f'Cannot decode {row.full_image}')
            roi = make_square_roi(
                image,
                [row.x1, row.y1, row.x2, row.y2],
                PRODUCTION_EXPANSION,
                0.0,
            )
            rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
            features = extract_glcm_features(rgb)
            item = {'split': row.split, 'patient': row.patient, 'side': row.side}
            item.update(
                {
                    f'glcm_{name.lower()}': float(value)
                    for name, value in zip(GLCM_PROPERTIES, features)
                }
            )
            rows.append(item)
        cache = pd.DataFrame(rows)
        GLCM_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        cache.to_csv(GLCM_CACHE_PATH, index=False)

    feature_columns = [f'glcm_{name.lower()}' for name in GLCM_PROPERTIES]
    required = {'split', 'patient', 'side', *feature_columns}
    if not required.issubset(cache.columns):
        raise RuntimeError(f'Invalid GLCM cache: {GLCM_CACHE_PATH}')
    if cache.duplicated(['split', 'patient', 'side']).any():
        raise RuntimeError('GLCM cache contains duplicate keys')
    return cache, feature_columns


glcm_cache, GLCM_COLUMNS = build_or_load_glcm_cache(manifest)
manifest = manifest.merge(
    glcm_cache,
    on=['split', 'patient', 'side'],
    how='left',
    validate='one_to_one',
)
if manifest[GLCM_COLUMNS].isna().any().any():
    raise RuntimeError('Missing GLCM features after manifest merge')

train_mask = manifest.split == 'train'
glcm_mean = manifest.loc[train_mask, GLCM_COLUMNS].mean().to_numpy(np.float32)
glcm_std = manifest.loc[train_mask, GLCM_COLUMNS].std(ddof=0).to_numpy(np.float32)
glcm_std = np.where(glcm_std > 1e-8, glcm_std, 1.0).astype(np.float32)
GLCM_Z_COLUMNS = [f'{column}_z' for column in GLCM_COLUMNS]
manifest[GLCM_Z_COLUMNS] = (
    manifest[GLCM_COLUMNS].to_numpy(np.float32) - glcm_mean
) / glcm_std
if not np.isfinite(manifest[GLCM_Z_COLUMNS].to_numpy()).all():
    raise RuntimeError('Non-finite standardized GLCM feature detected')

glcm_stats = {
    'feature_columns': GLCM_COLUMNS,
    'mean': glcm_mean.tolist(),
    'std': glcm_std.tolist(),
    'texture_size': [TEXTURE_HEIGHT, TEXTURE_WIDTH],
    'levels': GLCM_LEVELS,
    'distances': list(GLCM_DISTANCES),
    'angles_radians': list(GLCM_ANGLES),
    'symmetric': True,
    'normed': True,
}
(RUN_DIR / 'glcm_stats.json').write_text(json.dumps(glcm_stats, indent=2))
display(manifest.loc[train_mask, GLCM_COLUMNS].describe().T)


train_transform = transforms.Compose(
    [
        OpenCVCLAHE(),
        SquarePad(),
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(p=0.50),
        transforms.RandomRotation(5),
        transforms.ColorJitter(brightness=0.08, contrast=0.08),
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.RandomErasing(
            p=0.10,
            scale=(0.02, 0.05),
            ratio=(0.5, 2.0),
            value=0,
        ),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225],
        ),
    ]
)
val_transform = transforms.Compose(
    [
        OpenCVCLAHE(),
        SquarePad(),
        transforms.ToPILImage(),
        transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.485, 0.456, 0.406],
            [0.229, 0.224, 0.225],
        ),
    ]
)


class ProductionGLCMDataset(Dataset):
    def __init__(self, frame, transform, training):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.training = training
        self.labels = self.frame.grade.astype(int).tolist()

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = cv2.imread(row.full_image, cv2.IMREAD_COLOR)
        if image is None:
            raise RuntimeError(f'Cannot decode {row.full_image}')
        expansion = (
            random.uniform(*TRAIN_EXPANSION_RANGE)
            if self.training
            else PRODUCTION_EXPANSION
        )
        shift = TRAIN_SHIFT_FRACTION if self.training else 0.0
        roi = make_square_roi(
            image,
            [row.x1, row.y1, row.x2, row.y2],
            expansion,
            shift,
        )
        rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
        glcm = torch.as_tensor(
            row[GLCM_Z_COLUMNS].to_numpy(dtype=np.float32),
            dtype=torch.float32,
        )
        return (
            self.transform(rgb),
            glcm,
            int(row.grade),
            str(row.patient),
            str(row.side),
        )


train_frame = manifest.loc[manifest.split == 'train'].copy()
val_frame = manifest.loc[manifest.split == 'val'].copy()


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def build_loaders(seed):
    train_data = ProductionGLCMDataset(train_frame, train_transform, training=True)
    val_data = ProductionGLCMDataset(val_frame, val_transform, training=False)
    counts = np.bincount(train_frame.grade.to_numpy(), minlength=5)
    sample_weights = (1.0 / counts)[train_frame.grade.to_numpy()]
    sampler_generator = torch.Generator().manual_seed(seed)
    loader_generator = torch.Generator().manual_seed(seed + 10000)
    sampler = WeightedRandomSampler(
        torch.as_tensor(sample_weights, dtype=torch.double),
        len(sample_weights),
        replacement=True,
        generator=sampler_generator,
    )
    common = {
        'num_workers': NUM_WORKERS,
        'pin_memory': DEVICE.type == 'cuda',
        'persistent_workers': NUM_WORKERS > 0,
        'worker_init_fn': seed_worker,
        'generator': loader_generator,
    }
    train_loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        **common,
    )
    val_loader = DataLoader(
        val_data,
        batch_size=BATCH_SIZE,
        shuffle=False,
        **common,
    )
    return train_loader, val_loader



GLCM:   0%|          | 0/6604 [00:00<?, ?it/s]

,count,mean,std,min,25%,50%,75%,max
glcm_contrast,5778.0,3.088228,1.535273,0.282693,1.856294,2.900923,4.152603,9.886671
glcm_dissimilarity,5778.0,1.077709,0.321712,0.234877,0.828169,1.089831,1.328490,1.904355
glcm_homogeneity,5778.0,0.608131,0.083858,0.446135,0.538854,0.593801,0.667461,0.890511
glcm_energy,5778.0,0.133945,0.045063,0.077549,0.104382,0.122617,0.147597,0.518568
glcm_correlation,5778.0,0.952183,0.030095,0.775390,0.928214,0.959803,0.978059,0.995663
glcm_asm,5778.0,0.020210,0.017009,0.006217,0.011113,0.015253,0.022053,0.269016


In [ ]:
class DenseNet121GLCMModel(nn.Module):
    def __init__(self, use_glcm):
        super().__init__()
        self.use_glcm = bool(use_glcm)
        self.backbone = timm.create_model(
            'densenet121',
            pretrained=False,
            num_classes=5,
            drop_rate=0.20,
        )
        self.glcm_branch = nn.Sequential(
            nn.Linear(6, 16),
            nn.ReLU(inplace=True),
            nn.Dropout(0.20),
            nn.Linear(16, 5),
        )
        self.alpha_logit = nn.Parameter(torch.tensor(-4.0))
        if not self.use_glcm:
            self.glcm_branch.requires_grad_(False)
            self.alpha_logit.requires_grad_(False)

    @property
    def gradcam_target_layer(self):
        return self.backbone.features.norm5

    def forward_components(self, images, glcm):
        cnn_logits = self.backbone(images)
        if self.use_glcm:
            scaled_glcm_logits = torch.sigmoid(self.alpha_logit) * self.glcm_branch(glcm)
        else:
            scaled_glcm_logits = torch.zeros_like(cnn_logits)
        return cnn_logits + scaled_glcm_logits, cnn_logits, scaled_glcm_logits

    def forward(self, images, glcm):
        return self.forward_components(images, glcm)[0]


def checkpoint_state(checkpoint):
    state = checkpoint.get('model_state_dict')
    if state is None:
        state = checkpoint.get('model')
    if not isinstance(state, dict):
        raise RuntimeError('Checkpoint has no model_state_dict or model mapping')
    return state


def load_from_base(use_glcm):
    checkpoint = torch.load(BASE_CHECKPOINT, map_location='cpu', weights_only=False)
    if checkpoint.get('loss_type') not in (None, 'ce'):
        raise RuntimeError(f'Expected CE checkpoint; got {checkpoint.get("loss_type")}')
    model = DenseNet121GLCMModel(use_glcm=use_glcm)
    incompatible = model.load_state_dict(checkpoint_state(checkpoint), strict=False)
    allowed_missing = {
        'alpha_logit',
        'glcm_branch.0.weight',
        'glcm_branch.0.bias',
        'glcm_branch.3.weight',
        'glcm_branch.3.bias',
    }
    if set(incompatible.missing_keys) != allowed_missing:
        raise RuntimeError(f'Unexpected missing checkpoint keys: {incompatible.missing_keys}')
    if incompatible.unexpected_keys:
        raise RuntimeError(f'Unexpected checkpoint keys: {incompatible.unexpected_keys}')
    return model.to(DEVICE), checkpoint


def load_selected(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model = DenseNet121GLCMModel(use_glcm=bool(checkpoint['use_glcm']))
    model.load_state_dict(checkpoint_state(checkpoint), strict=True)
    return model.to(DEVICE), checkpoint


def expected_calibration_error(labels, probabilities, bins=15):
    predictions = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    correct = predictions == labels
    edges = np.linspace(0.0, 1.0, bins + 1)
    result = 0.0
    for lower, upper in zip(edges[:-1], edges[1:]):
        selected = (confidence > lower) & (confidence <= upper)
        if selected.any():
            result += selected.mean() * abs(
                correct[selected].mean() - confidence[selected].mean()
            )
    return float(result)


def calculate_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = probabilities.argmax(axis=1)
    one_hot = np.eye(5, dtype=float)[labels]
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        labels=list(range(5)),
        average=None,
        zero_division=0,
    )
    return {
        'accuracy': float(np.mean(labels == predictions)),
        'balanced_accuracy': float(balanced_accuracy_score(labels, predictions)),
        'qwk': float(cohen_kappa_score(labels, predictions, weights='quadratic')),
        'mae': float(np.mean(np.abs(labels - predictions))),
        'macro_precision': float(np.mean(precision)),
        'macro_recall': float(np.mean(recall)),
        'macro_f1': float(np.mean(f1)),
        'macro_ap': float(average_precision_score(one_hot, probabilities, average='macro')),
        'macro_auc': float(
            roc_auc_score(one_hot, probabilities, average='macro', multi_class='ovr')
        ),
        'grade1_recall': float(recall[1]),
        'grade4_recall': float(recall[4]),
        'adjacent_error_rate': float(
            np.mean((labels != predictions) & (np.abs(labels - predictions) == 1))
        ),
        'severe_error_rate': float(np.mean(np.abs(labels - predictions) >= 2)),
        'nll': float(-np.mean(np.log(np.clip(probabilities[np.arange(len(labels)), labels], 1e-8, 1.0)))),
        'brier': float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1))),
        'ece': expected_calibration_error(labels, probabilities),
    }


def evaluate(model, loader, prediction_path=None, report_path=None):
    labels, probabilities, records = [], [], []
    model.eval()
    with torch.inference_mode():
        for images, glcm, batch_labels, patients, sides in loader:
            images = images.to(DEVICE, non_blocking=True)
            glcm = glcm.to(DEVICE, non_blocking=True)
            final_logits, cnn_logits, glcm_logits = model.forward_components(images, glcm)
            probs = F.softmax(final_logits.float(), dim=1).cpu().numpy()
            cnn_values = cnn_logits.float().cpu().numpy()
            glcm_values = glcm_logits.float().cpu().numpy()
            for index in range(len(batch_labels)):
                item = {
                    'patient': str(patients[index]),
                    'side': str(sides[index]),
                    'true_grade': int(batch_labels[index]),
                    'predicted_grade': int(probs[index].argmax()),
                }
                for grade in range(5):
                    item[f'prob_grade_{grade}'] = float(probs[index, grade])
                    item[f'cnn_logit_{grade}'] = float(cnn_values[index, grade])
                    item[f'glcm_logit_{grade}'] = float(glcm_values[index, grade])
                records.append(item)
            labels.extend(batch_labels.numpy())
            probabilities.extend(probs)

    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    metrics = calculate_metrics(labels, probabilities)
    predictions = probabilities.argmax(axis=1)
    if prediction_path is not None:
        pd.DataFrame(records).to_csv(prediction_path, index=False)
    if report_path is not None:
        report = classification_report(
            labels,
            predictions,
            labels=list(range(5)),
            output_dict=True,
            zero_division=0,
        )
        pd.DataFrame(report).transpose().to_csv(report_path)
    return metrics, labels, probabilities


def save_confusion(labels, probabilities, output_path, title):
    predictions = probabilities.argmax(axis=1)
    matrix = confusion_matrix(labels, predictions, labels=list(range(5)))
    figure, axis = plt.subplots(figsize=(6, 5))
    image = axis.imshow(matrix, cmap='Blues')
    for row in range(5):
        for column in range(5):
            axis.text(column, row, str(matrix[row, column]), ha='center', va='center')
    axis.set(
        xlabel='Predicted KL grade',
        ylabel='True KL grade',
        xticks=range(5),
        yticks=range(5),
        title=title,
    )
    figure.colorbar(image, ax=axis)
    figure.tight_layout()
    figure.savefig(output_path, dpi=160)
    plt.close(figure)


def train_one(arm, use_glcm, seed):
    seed_everything(seed)
    arm_dir = RUN_DIR / arm / f'seed_{seed}'
    arm_dir.mkdir(parents=True, exist_ok=False)
    train_loader, val_loader = build_loaders(seed)
    model, base_metadata = load_from_base(use_glcm=use_glcm)

    backbone_parameters = [
        parameter for parameter in model.backbone.parameters() if parameter.requires_grad
    ]
    parameter_groups = [{'params': backbone_parameters, 'lr': BACKBONE_LR}]
    if use_glcm:
        parameter_groups.append(
            {
                'params': [*model.glcm_branch.parameters(), model.alpha_logit],
                'lr': GLCM_BRANCH_LR,
            }
        )
    optimizer = torch.optim.AdamW(
        parameter_groups,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
        eta_min=1e-7,
    )
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    history, best_score = [], -float('inf')

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, samples = 0.0, 0
        for images, glcm, labels, _, _ in tqdm(
            train_loader,
            desc=f'{arm} seed={seed} epoch={epoch}/{EPOCHS}',
        ):
            images = images.to(DEVICE, non_blocking=True)
            glcm = glcm.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=DEVICE.type == 'cuda'):
                loss = F.cross_entropy(model(images, glcm), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * labels.size(0)
            samples += labels.size(0)
        scheduler.step()

        metrics, _, _ = evaluate(model, val_loader)
        score = (
            0.55 * metrics['qwk']
            + 0.30 * metrics['macro_f1']
            + 0.15 * metrics['macro_ap']
        )
        row = {
            'arm': arm,
            'seed': seed,
            'epoch': epoch,
            'train_loss': loss_sum / max(samples, 1),
            'selection_score': score,
            'alpha': float(torch.sigmoid(model.alpha_logit).item()) if use_glcm else 0.0,
            **metrics,
        }
        history.append(row)
        print(json.dumps(row, indent=2))
        if score > best_score:
            best_score = score
            torch.save(
                {
                    'model_state_dict': model.state_dict(),
                    'architecture': 'timm_densenet121_additive_glcm_v1',
                    'model_name': 'densenet121',
                    'use_glcm': use_glcm,
                    'loss_type': 'ce',
                    'epoch': epoch,
                    'seed': seed,
                    'arm': arm,
                    'base_checkpoint': str(BASE_CHECKPOINT),
                    'base_architecture': base_metadata.get('architecture'),
                    'yolo_checkpoint': str(YOLO_CHECKPOINT),
                    'glcm_stats': glcm_stats,
                    'validation_metrics': metrics,
                    'selection_score': score,
                    'alpha': float(torch.sigmoid(model.alpha_logit).item()) if use_glcm else 0.0,
                },
                arm_dir / 'best_model.pth',
            )

    history_frame = pd.DataFrame(history)
    history_frame.to_csv(arm_dir / 'history.csv', index=False)
    selected_model, selected_metadata = load_selected(arm_dir / 'best_model.pth')
    final_metrics, labels, probabilities = evaluate(
        selected_model,
        val_loader,
        prediction_path=arm_dir / 'validation_predictions.csv',
        report_path=arm_dir / 'classification_report.csv',
    )
    final_metrics.update(
        {
            'arm': arm,
            'seed': seed,
            'selected_epoch': int(selected_metadata['epoch']),
            'selection_score': float(selected_metadata['selection_score']),
            'alpha': float(selected_metadata['alpha']),
        }
    )
    (arm_dir / 'metrics.json').write_text(json.dumps(final_metrics, indent=2))
    save_confusion(
        labels,
        probabilities,
        arm_dir / 'confusion_matrix.png',
        f'{arm}, seed {seed}',
    )

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history_frame.epoch, history_frame.train_loss, marker='o')
    axes[0].set(title='Training loss', xlabel='Epoch', ylabel='CE loss')
    axes[1].plot(history_frame.epoch, history_frame.selection_score, marker='o')
    axes[1].set(title='Validation selection score', xlabel='Epoch', ylabel='Score')
    figure.tight_layout()
    figure.savefig(arm_dir / 'training_curves.png', dpi=160)
    plt.close(figure)

    del model, selected_model, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return final_metrics, history_frame



In [ ]:
run_config = {
    'experiment': 'production_aligned_densenet121_glcm_additive_comparison',
    'base_checkpoint': str(BASE_CHECKPOINT),
    'yolo_checkpoint': str(YOLO_CHECKPOINT),
    'manifest': str(MANIFEST_PATH),
    'glcm_cache': str(GLCM_CACHE_PATH),
    'input_size': INPUT_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'backbone_lr': BACKBONE_LR,
    'glcm_branch_lr': GLCM_BRANCH_LR,
    'weight_decay': WEIGHT_DECAY,
    'seeds': list(SEEDS),
    'loss': 'unweighted_cross_entropy',
    'sampler': 'full_inverse_frequency',
    'checkpoint_selection': '0.55*QWK + 0.30*macro_F1 + 0.15*macro_AP',
    'production_roi_expansion': PRODUCTION_EXPANSION,
    'training_roi_expansion': list(TRAIN_EXPANSION_RANGE),
    'training_roi_shift_fraction': TRAIN_SHIFT_FRACTION,
    'preprocessing': 'CLAHE 1.25 -> SquarePad -> Resize 384 -> ImageNet normalize',
    'glcm': glcm_stats,
    'arms': {
        'deep_control': {'use_glcm': False},
        'deep_glcm_additive': {'use_glcm': True},
    },
}
(RUN_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2))

all_metrics, all_history = [], []
for seed in SEEDS:
    for arm, use_glcm in (
        ('deep_control', False),
        ('deep_glcm_additive', True),
    ):
        metrics, history = train_one(arm, use_glcm, seed)
        all_metrics.append(metrics)
        all_history.append(history)

metrics_frame = pd.DataFrame(all_metrics)
history_frame = pd.concat(all_history, ignore_index=True)
metrics_frame.to_csv(RUN_DIR / 'metrics_by_seed.csv', index=False)
history_frame.to_csv(RUN_DIR / 'history_all.csv', index=False)
display(metrics_frame)



deep_control seed=42 epoch=1/5:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "arm": "deep_control",
  "seed": 42,
  "epoch": 1,
  "train_loss": 0.7624503718234544,
  "selection_score": 0.7070805797963994,
  "alpha": 0.0,
  "accuracy": 0.6016949152542372,
  "balanced_accuracy": 0.6383426474849285,
  "qwk": 0.7567699141970425,
  "mae": 0.4927360774818402,
  "macro_precision": 0.6240549844048376,
  "macro_recall": 0.6383426474849285,
  "macro_f1": 0.6299597146786106,
  "macro_ap": 0.6791280838962852,
  "macro_auc": 0.8690614082903567,
  "grade1_recall": 0.35947712418300654,
  "grade4_recall": 0.8518518518518519,
  "adjacent_error_rate": 0.30871670702179177,
  "severe_error_rate": 0.08958837772397095,
  "nll": 0.9045488791334604,
  "brier": 0.5076764700591405,
  "ece": 0.07120323729572804
}


deep_control seed=42 epoch=2/5:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "arm": "deep_control",
  "seed": 42,
  "epoch": 2,
  "train_loss": 0.7533296980466674,
  "selection_score": 0.7165646933987269,
  "alpha": 0.0,
  "accuracy": 0.6198547215496368,
  "balanced_accuracy": 0.6539241753867779,
  "qwk": 0.7633850425544758,
  "mae": 0.4745762711864407,
  "macro_precision": 0.6392830027735577,
  "macro_recall": 0.6539241753867779,
  "macro_f1": 0.644695354238771,
  "macro_ap": 0.6886287581475585,
  "macro_auc": 0.8714051937403582,
  "grade1_recall": 0.40522875816993464,
  "grade4_recall": 0.8518518518518519,
  "adjacent_error_rate": 0.2929782082324455,
  "severe_error_rate": 0.08716707021791767,
  "nll": 0.9022728878421107,
  "brier": 0.5066101735728066,
  "ece": 0.048457531461415515
}


deep_control seed=42 epoch=3/5:   0%|          | 0/121 [00:00<?, ?it/s]

...

deep_glcm_additive seed=2026 epoch=4/5:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "arm": "deep_glcm_additive",
  "seed": 2026,
  "epoch": 4,
  "train_loss": 0.7295527492851979,
  "selection_score": 0.720800624324459,
  "alpha": 0.017950300127267838,
  "accuracy": 0.6222760290556901,
  "balanced_accuracy": 0.6570475492953228,
  "qwk": 0.7678473449978662,
  "mae": 0.46731234866828086,
  "macro_precision": 0.6412889180607687,
  "macro_recall": 0.6570475492953228,
  "macro_f1": 0.6471849331469397,
  "macro_ap": 0.6955273642103379,
  "macro_auc": 0.8732565676651547,
  "grade1_recall": 0.43137254901960786,
  "grade4_recall": 0.8518518518518519,
  "adjacent_error_rate": 0.29418886198547217,
  "severe_error_rate": 0.08353510895883777,
  "nll": 0.8921743896460514,
  "brier": 0.5029016451671243,
  "ece": 0.06002135390808163
}


deep_glcm_additive seed=2026 epoch=5/5:   0%|          | 0/121 [00:00<?, ?it/s]

{
  "arm": "deep_glcm_additive",
  "seed": 2026,
  "epoch": 5,
  "train_loss": 0.7270418783090815,
  "selection_score": 0.7255476170121608,
  "alpha": 0.017951903864741325,
  "accuracy": 0.6234866828087167,
  "balanced_accuracy": 0.657353818451244,
  "qwk": 0.7736800049173275,
  "mae": 0.4600484261501211,
  "macro_precision": 0.6505832452499136,
  "macro_recall": 0.657353818451244,
  "macro_f1": 0.6519083150712828,
  "macro_ap": 0.6963407985749721,
  "macro_auc": 0.8739447219587309,
  "grade1_recall": 0.43790849673202614,
  "grade4_recall": 0.8518518518518519,
  "adjacent_error_rate": 0.29782082324455206,
  "severe_error_rate": 0.07869249394673124,
  "nll": 0.886146721924235,
  "brier": 0.5010516291559868,
  "ece": 0.05641435065078966
}


,accuracy,balanced_accuracy,qwk,mae,macro_precision,macro_recall,macro_f1,macro_ap,macro_auc,grade1_recall,...,adjacent_error_rate,severe_error_rate,nll,brier,ece,arm,seed,selected_epoch,selection_score,alpha
0,0.623487,0.662911,0.763668,0.468523,0.647973,0.662911,0.653841,0.685976,0.871279,0.437908,...,0.290557,0.085956,0.898979,0.506233,0.052144,deep_control,42,3,0.719066,0.000000
1,0.617433,0.648274,0.763808,0.475787,0.638871,0.648274,0.641737,0.685649,0.870157,0.372549,...,0.296610,0.085956,0.891301,0.503243,0.047626,deep_glcm_additive,42,4,0.715463,0.017959
2,0.636804,0.662272,0.777059,0.451574,0.653226,0.662272,0.656686,0.692329,0.872779,0.411765,...,0.279661,0.083535,0.884071,0.496048,0.042229,deep_control,1337,5,0.728238,0.000000
3,0.644068,0.664987,0.782711,0.440678,0.652944,0.664987,0.657587,0.692518,0.873742,0.411765,...,0.278450,0.077482,0.886391,0.493863,0.052664,deep_glcm_additive,1337,3,0.731645,0.018005
4,0.641646,0.661978,0.775114,0.447942,0.657506,0.661978,0.658832,0.700864,0.875684,0.424837,...,0.274818,0.083535,0.873876,0.492454,0.052952,deep_control,2026,3,0.729092,0.000000
5,0.623487,0.657354,0.773680,0.460048,0.650583,0.657354,0.651908,0.696341,0.873945,0.437908,...,0.297821,0.078692,0.886147,0.501052,0.056414,deep_glcm_additive,2026,5,0.725548,0.017952


In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.activation = None
        self.handle = model.gradcam_target_layer.register_forward_hook(self._capture)

    def _capture(self, module, inputs, output):
        self.activation = output

    def remove(self):
        self.handle.remove()

    def __call__(self, image, glcm):
        self.model.eval()
        self.model.zero_grad(set_to_none=True)
        self.activation = None
        with torch.enable_grad():
            final_logits, cnn_logits, scaled_glcm_logits = self.model.forward_components(
                image.detach().clone().requires_grad_(True),
                glcm,
            )
            target = int(final_logits.argmax(dim=1).item())
            gradient = torch.autograd.grad(
                cnn_logits[0, target],
                self.activation,
                retain_graph=False,
                create_graph=False,
            )[0]
            weights = gradient.mean(dim=(2, 3), keepdim=True)
            cam = F.relu((weights * self.activation).sum(dim=1, keepdim=True))
            cam = F.interpolate(
                cam,
                size=image.shape[-2:],
                mode='bilinear',
                align_corners=False,
            )[0, 0]
        cam = cam.detach().float().cpu().numpy()
        maximum = float(cam.max())
        cam = cam / maximum if maximum > 1e-8 else np.zeros_like(cam)
        probabilities = F.softmax(final_logits.float(), dim=1)[0].detach().cpu().numpy()
        return (
            target,
            probabilities,
            cnn_logits[0].detach().cpu().numpy(),
            scaled_glcm_logits[0].detach().cpu().numpy(),
            cam,
        )


def anatomy_masks(height, width):
    joint = np.zeros((height, width), dtype=bool)
    joint[int(0.28 * height) : int(0.72 * height), int(0.06 * width) : int(0.94 * width)] = True
    border = np.ones((height, width), dtype=bool)
    border[int(0.08 * height) : int(0.92 * height), int(0.08 * width) : int(0.92 * width)] = False
    lower_tibia = np.zeros((height, width), dtype=bool)
    lower_tibia[int(0.72 * height) : int(0.96 * height), int(0.06 * width) : int(0.94 * width)] = True
    return joint, border, lower_tibia


def cam_measurements(cam):
    height, width = cam.shape
    joint, border, lower_tibia = anatomy_masks(height, width)
    total = float(cam.sum()) + 1e-8
    peak_y, peak_x = np.unravel_index(int(np.argmax(cam)), cam.shape)
    joint_energy = float(cam[joint].sum() / total)
    border_energy = float(cam[border].sum() / total)
    lower_tibia_energy = float(cam[lower_tibia].sum() / total)
    peak_inside = bool(joint[peak_y, peak_x] and total > 1e-7)
    gate_pass = bool(
        joint_energy >= 0.55
        and border_energy <= 0.25
        and lower_tibia_energy <= 0.25
        and peak_inside
    )
    return {
        'joint_energy': joint_energy,
        'joint_enrichment': joint_energy / float(joint.mean()),
        'border_energy': border_energy,
        'border_enrichment': border_energy / float(border.mean()),
        'lower_tibia_energy': lower_tibia_energy,
        'peak_inside_joint': peak_inside,
        'gate_pass': gate_pass,
    }


def occlusion_drops(model, image, glcm, target):
    height, width = image.shape[-2:]
    joint, border, _ = anatomy_masks(height, width)
    joint_mask = torch.from_numpy(joint).to(image.device)[None, None]
    border_mask = torch.from_numpy(border).to(image.device)[None, None]
    joint_image = torch.where(joint_mask, torch.zeros_like(image), image)
    border_image = torch.where(border_mask, torch.zeros_like(image), image)
    with torch.inference_mode():
        original = F.softmax(model(image, glcm).float(), dim=1)[0, target]
        joint_probability = F.softmax(model(joint_image, glcm).float(), dim=1)[0, target]
        border_probability = F.softmax(model(border_image, glcm).float(), dim=1)[0, target]
    joint_drop = float((original - joint_probability).item())
    border_drop = float((original - border_probability).item())
    return {
        'joint_occlusion_drop': joint_drop,
        'border_occlusion_drop': border_drop,
        'joint_minus_border_drop': joint_drop - border_drop,
    }


def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    image = (tensor.detach().cpu() * std + mean).clamp(0, 1)
    return np.uint8(image.permute(1, 2, 0).numpy() * 255)


def render_overlay(image_rgb, cam):
    heatmap = cv2.applyColorMap(np.uint8(np.clip(cam, 0, 1) * 255), cv2.COLORMAP_TURBO)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(image_rgb, 0.65, heatmap, 0.35, 0)


audit_parts = []
for grade in range(5):
    grade_frame = val_frame.loc[val_frame.grade == grade]
    count = min(CAM_CASES_PER_GRADE, len(grade_frame))
    audit_parts.append(grade_frame.sample(n=count, random_state=2026 + grade))
audit_frame = pd.concat(audit_parts, ignore_index=True)
audit_data = ProductionGLCMDataset(audit_frame, val_transform, training=False)


def audit_checkpoint(arm, seed):
    checkpoint_path = RUN_DIR / arm / f'seed_{seed}' / 'best_model.pth'
    model, metadata = load_selected(checkpoint_path)
    engine = GradCAM(model)
    rows = []
    example_counts = {grade: 0 for grade in range(5)}
    example_dir = RUN_DIR / 'gradcam_examples' / arm / f'seed_{seed}'
    example_dir.mkdir(parents=True, exist_ok=True)
    for index in tqdm(range(len(audit_data)), desc=f'CAM {arm} seed={seed}'):
        tensor, glcm, true_grade, patient, side = audit_data[index]
        image = tensor.unsqueeze(0).to(DEVICE)
        glcm_batch = glcm.unsqueeze(0).to(DEVICE)
        target, probabilities, cnn_logits, glcm_logits, cam = engine(image, glcm_batch)
        measures = cam_measurements(cam)
        drops = occlusion_drops(model, image, glcm_batch, target)
        contribution_ratio = float(
            abs(glcm_logits[target])
            / (abs(cnn_logits[target]) + abs(glcm_logits[target]) + 1e-8)
        )
        rows.append(
            {
                'arm': arm,
                'seed': seed,
                'patient': patient,
                'side': side,
                'true_grade': true_grade,
                'predicted_grade': target,
                'confidence': float(probabilities[target]),
                'alpha': float(metadata['alpha']),
                'glcm_winning_logit_fraction': contribution_ratio,
                **measures,
                **drops,
            }
        )
        if (
            seed == SEEDS[0]
            and example_counts[true_grade] < CAM_EXAMPLES_PER_GRADE
        ):
            processed = denormalize(tensor)
            figure, axes = plt.subplots(1, 2, figsize=(9, 4))
            axes[0].imshow(processed, cmap='gray')
            axes[0].set_title(f'True KL-{true_grade}')
            axes[1].imshow(render_overlay(processed, cam))
            axes[1].set_title(
                f'Pred KL-{target}, joint={measures["joint_energy"]:.2f}'
            )
            for axis in axes:
                axis.axis('off')
            figure.tight_layout()
            figure.savefig(
                example_dir / f'G{true_grade}_{patient}_{side}.png',
                dpi=160,
            )
            plt.close(figure)
            example_counts[true_grade] += 1
    engine.remove()
    result = pd.DataFrame(rows)
    result.to_csv(
        RUN_DIR / arm / f'seed_{seed}' / 'gradcam_audit.csv',
        index=False,
    )
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


all_cam_rows = []
for seed in SEEDS:
    for arm in ('deep_control', 'deep_glcm_additive'):
        all_cam_rows.append(audit_checkpoint(arm, seed))
cam_frame = pd.concat(all_cam_rows, ignore_index=True)
cam_frame.to_csv(RUN_DIR / 'gradcam_audit_all.csv', index=False)
cam_metric_columns = [
    'joint_energy',
    'joint_enrichment',
    'border_energy',
    'border_enrichment',
    'lower_tibia_energy',
    'peak_inside_joint',
    'gate_pass',
    'joint_occlusion_drop',
    'border_occlusion_drop',
    'joint_minus_border_drop',
    'glcm_winning_logit_fraction',
]
cam_summary = (
    cam_frame.groupby(['arm', 'seed'])[cam_metric_columns]
    .mean()
    .reset_index()
)
cam_summary.to_csv(RUN_DIR / 'gradcam_summary_by_seed.csv', index=False)
display(cam_summary)



CAM deep_control seed=42:   0%|          | 0/100 [00:00<?, ?it/s]

CAM deep_glcm_additive seed=42:   0%|          | 0/100 [00:00<?, ?it/s]

CAM deep_control seed=1337:   0%|          | 0/100 [00:00<?, ?it/s]

CAM deep_glcm_additive seed=1337:   0%|          | 0/100 [00:00<?, ?it/s]

CAM deep_control seed=2026:   0%|          | 0/100 [00:00<?, ?it/s]

CAM deep_glcm_additive seed=2026:   0%|          | 0/100 [00:00<?, ?it/s]

,arm,seed,joint_energy,joint_enrichment,border_energy,border_enrichment,lower_tibia_energy,peak_inside_joint,gate_pass,joint_occlusion_drop,border_occlusion_drop,joint_minus_border_drop,glcm_winning_logit_fraction
0,deep_control,42,0.855425,2.214765,0.081688,0.279299,0.070180,1.0,1.0,0.484767,0.238550,0.246217,0.000000
1,deep_control,1337,0.855829,2.215812,0.082125,0.280793,0.070762,1.0,1.0,0.469414,0.111193,0.358221,0.000000
2,deep_control,2026,0.858307,2.222228,0.080096,0.273857,0.069004,1.0,1.0,0.480948,0.188272,0.292677,0.000000
3,deep_glcm_additive,42,0.858021,2.221488,0.080207,0.274235,0.068215,1.0,1.0,0.485928,0.160667,0.325261,0.002095
4,deep_glcm_additive,1337,0.855273,2.214372,0.083423,0.285232,0.070951,1.0,1.0,0.472094,0.183568,0.288526,0.002355
5,deep_glcm_additive,2026,0.855194,2.214169,0.082011,0.280404,0.070164,1.0,1.0,0.473427,0.183086,0.290342,0.001277


In [ ]:
def paired_patient_bootstrap(seed, resamples=BOOTSTRAP_RESAMPLES):
    control = pd.read_csv(
        RUN_DIR / 'deep_control' / f'seed_{seed}' / 'validation_predictions.csv',
        dtype={'patient': str, 'side': str},
    )
    hybrid = pd.read_csv(
        RUN_DIR / 'deep_glcm_additive' / f'seed_{seed}' / 'validation_predictions.csv',
        dtype={'patient': str, 'side': str},
    )
    merged = control.merge(
        hybrid,
        on=['patient', 'side', 'true_grade'],
        suffixes=('_control', '_hybrid'),
        validate='one_to_one',
    )
    patient_indices = {
        patient: np.flatnonzero(merged.patient.to_numpy() == patient)
        for patient in merged.patient.unique()
    }
    patients = np.asarray(list(patient_indices))
    rng = np.random.default_rng(seed + 70000)
    rows = []
    for _ in range(resamples):
        sampled = rng.choice(patients, size=len(patients), replace=True)
        indices = np.concatenate([patient_indices[patient] for patient in sampled])
        labels = merged.true_grade.to_numpy()[indices]
        control_predictions = merged.predicted_grade_control.to_numpy()[indices]
        hybrid_predictions = merged.predicted_grade_hybrid.to_numpy()[indices]
        rows.append(
            {
                'qwk_delta': cohen_kappa_score(
                    labels, hybrid_predictions, weights='quadratic'
                )
                - cohen_kappa_score(labels, control_predictions, weights='quadratic'),
                'macro_f1_delta': f1_score(
                    labels,
                    hybrid_predictions,
                    labels=list(range(5)),
                    average='macro',
                    zero_division=0,
                )
                - f1_score(
                    labels,
                    control_predictions,
                    labels=list(range(5)),
                    average='macro',
                    zero_division=0,
                ),
            }
        )
    samples = pd.DataFrame(rows)
    summary = {'seed': seed}
    for column in samples.columns:
        summary[f'{column}_low'] = float(samples[column].quantile(0.025))
        summary[f'{column}_median'] = float(samples[column].median())
        summary[f'{column}_high'] = float(samples[column].quantile(0.975))
    samples.to_csv(RUN_DIR / f'paired_bootstrap_seed_{seed}.csv', index=False)
    return summary


bootstrap_summary = pd.DataFrame(
    [paired_patient_bootstrap(seed) for seed in SEEDS]
)
bootstrap_summary.to_csv(RUN_DIR / 'paired_bootstrap_summary.csv', index=False)

metric_names = [
    'accuracy',
    'balanced_accuracy',
    'qwk',
    'mae',
    'macro_f1',
    'macro_ap',
    'macro_auc',
    'grade1_recall',
    'grade4_recall',
    'severe_error_rate',
    'nll',
    'brier',
    'ece',
    'alpha',
]
aggregate_metrics = metrics_frame.groupby('arm')[metric_names].agg(['mean', 'std'])
aggregate_metrics.columns = [f'{metric}_{stat}' for metric, stat in aggregate_metrics.columns]
aggregate_metrics = aggregate_metrics.reset_index()
aggregate_metrics.to_csv(RUN_DIR / 'metrics_aggregate.csv', index=False)

pivot = metrics_frame.pivot(index='seed', columns='arm', values=metric_names)
paired_deltas = pd.DataFrame(index=list(SEEDS))
for metric in metric_names:
    paired_deltas[f'{metric}_delta'] = (
        pivot[(metric, 'deep_glcm_additive')]
        - pivot[(metric, 'deep_control')]
    )
paired_deltas.index.name = 'seed'
paired_deltas.to_csv(RUN_DIR / 'paired_metric_deltas.csv')

cam_pivot = cam_summary.pivot(
    index='seed',
    columns='arm',
    values=cam_metric_columns,
)
cam_deltas = pd.DataFrame(index=list(SEEDS))
for metric in cam_metric_columns:
    cam_deltas[f'{metric}_delta'] = (
        cam_pivot[(metric, 'deep_glcm_additive')]
        - cam_pivot[(metric, 'deep_control')]
    )
cam_deltas.index.name = 'seed'
cam_deltas.to_csv(RUN_DIR / 'paired_cam_deltas.csv')

mean_metric_delta = paired_deltas.mean()
mean_cam_delta = cam_deltas.mean()
hybrid_cam = cam_summary.loc[cam_summary.arm == 'deep_glcm_additive']
criteria = {
    'qwk_gain_at_least_0_01': bool(mean_metric_delta.qwk_delta >= 0.01),
    'macro_f1_gain_at_least_0_01': bool(mean_metric_delta.macro_f1_delta >= 0.01),
    'macro_ap_no_material_regression': bool(mean_metric_delta.macro_ap_delta >= -0.005),
    'grade1_recall_drop_within_0_03': bool(mean_metric_delta.grade1_recall_delta >= -0.03),
    'severe_error_does_not_increase': bool(mean_metric_delta.severe_error_rate_delta <= 0.0),
    'ece_increase_within_0_01': bool(mean_metric_delta.ece_delta <= 0.01),
    'cam_gate_drop_less_than_0_05': bool(mean_cam_delta.gate_pass_delta > -0.05),
    'joint_occlusion_contrast_preserved': bool(
        mean_cam_delta.joint_minus_border_drop_delta >= -0.01
    ),
    'qwk_direction_consistent': bool((paired_deltas.qwk_delta > 0).all()),
    'macro_f1_direction_consistent': bool((paired_deltas.macro_f1_delta > 0).all()),
    'glcm_branch_not_dominant': bool(
        hybrid_cam.glcm_winning_logit_fraction.mean() <= 0.50
    ),
}
decision = {
    'promote_to_locked_holdout': bool(all(criteria.values())),
    'criteria': criteria,
}
(RUN_DIR / 'decision.json').write_text(json.dumps(decision, indent=2))

report_lines = [
    '# DenseNet-121 Production-Aligned GLCM Fusion Experiment',
    '',
    f'Run: `{RUN_TIMESTAMP}`',
    '',
    'This is a validation-only paired comparison. The repeatedly inspected test set was not used.',
    '',
    '## Configuration',
    '',
    pd.DataFrame(
        [
            ('Base checkpoint', str(BASE_CHECKPOINT)),
            ('YOLO checkpoint', str(YOLO_CHECKPOINT)),
            ('Input', f'{INPUT_SIZE} x {INPUT_SIZE}'),
            ('Epochs per arm/seed', EPOCHS),
            ('Seeds', ', '.join(map(str, SEEDS))),
            ('Loss', 'Unweighted five-class CE'),
            ('Imbalance', 'Full inverse-frequency sampler only'),
            ('GLCM', '6 features, q32, 104x224, distances 1/2/3, four angles'),
            ('Fusion', 'Additive class logits with learned constrained alpha'),
        ],
        columns=['Item', 'Value'],
    ).to_markdown(index=False),
    '',
    '## Validation Metrics by Seed',
    '',
    metrics_frame.to_markdown(index=False),
    '',
    '## Aggregate Metrics',
    '',
    aggregate_metrics.to_markdown(index=False),
    '',
    '## Paired GLCM Minus Control Deltas',
    '',
    paired_deltas.reset_index().to_markdown(index=False),
    '',
    '## Patient-Clustered Bootstrap Delta Intervals',
    '',
    bootstrap_summary.to_markdown(index=False),
    '',
    '## CAM Audit by Seed',
    '',
    cam_summary.to_markdown(index=False),
    '',
    '## Paired CAM Deltas',
    '',
    cam_deltas.reset_index().to_markdown(index=False),
    '',
    '## Acceptance Decision',
    '',
    f'Promote to a new locked holdout: **{decision["promote_to_locked_holdout"]}**',
    '',
    pd.DataFrame(
        [{'Criterion': name, 'Passed': passed} for name, passed in criteria.items()]
    ).to_markdown(index=False),
    '',
    '## Full Training History',
    '',
    history_frame.to_markdown(index=False),
    '',
    'Full per-grade classification reports, predictions, confusion matrices, training curves,',
    'CAM CSV files, and example overlays are stored in each arm/seed directory.',
]
(RUN_DIR / 'report.md').write_text('\n'.join(report_lines))

if NOTEBOOK_SOURCE.is_file():
    shutil.copy2(
        NOTEBOOK_SOURCE,
        RUN_DIR / f'{RUN_TIMESTAMP}_dense_net_121_glcm_fusion_comparison.ipynb',
    )
else:
    print('Notebook source was not found for automatic archival:', NOTEBOOK_SOURCE)

display(aggregate_metrics)
display(paired_deltas)
display(bootstrap_summary)
display(cam_summary)
print(json.dumps(decision, indent=2))
print('Complete run artifacts:', RUN_DIR)



Notebook source was not found for automatic archival: /content/dense_net_121_glcm_fusion_comparison.ipynb


,arm,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,qwk_mean,qwk_std,mae_mean,mae_std,macro_f1_mean,...,severe_error_rate_mean,severe_error_rate_std,nll_mean,nll_std,brier_mean,brier_std,ece_mean,ece_std,alpha_mean,alpha_std
0,deep_control,0.633979,0.009404,0.662387,0.000477,0.771947,0.007236,0.456013,0.010985,0.656453,...,0.084342,0.001398,0.885642,0.012625,0.498245,0.007148,0.049108,0.005971,0.000000,0.000000
1,deep_glcm_additive,0.628329,0.013962,0.656872,0.008367,0.773399,0.009455,0.458838,0.017586,0.650411,...,0.080710,0.004583,0.887946,0.002908,0.499386,0.004907,0.052234,0.004410,0.017972,0.000029


,accuracy_delta,balanced_accuracy_delta,qwk_delta,mae_delta,macro_f1_delta,macro_ap_delta,macro_auc_delta,grade1_recall_delta,grade4_recall_delta,severe_error_rate_delta,nll_delta,brier_delta,ece_delta,alpha_delta
seed,,,,,,,,,,,,,,
42,-0.006053,-0.014637,0.000140,0.007264,-0.012104,-0.000327,-0.001122,-0.065359,0.000000,0.000000,-0.007678,-0.002990,-0.004518,0.017959
1337,0.007264,0.002715,0.005651,-0.010896,0.000901,0.000189,0.000962,0.000000,0.000000,-0.006053,0.002319,-0.002185,0.010435,0.018005
2026,-0.018160,-0.004624,-0.001434,0.012107,-0.006924,-0.004523,-0.001739,0.013072,0.037037,-0.004843,0.012271,0.008597,0.003462,0.017952


,seed,qwk_delta_low,qwk_delta_median,qwk_delta_high,macro_f1_delta_low,macro_f1_delta_median,macro_f1_delta_high
0,42,-0.014829,-0.000051,0.014762,-0.032195,-0.012462,0.006018
1,1337,-0.006493,0.006056,0.018478,-0.018850,0.001282,0.020428
2,2026,-0.015803,-0.001230,0.012583,-0.028011,-0.007511,0.014137


,arm,seed,joint_energy,joint_enrichment,border_energy,border_enrichment,lower_tibia_energy,peak_inside_joint,gate_pass,joint_occlusion_drop,border_occlusion_drop,joint_minus_border_drop,glcm_winning_logit_fraction
0,deep_control,42,0.855425,2.214765,0.081688,0.279299,0.070180,1.0,1.0,0.484767,0.238550,0.246217,0.000000
1,deep_control,1337,0.855829,2.215812,0.082125,0.280793,0.070762,1.0,1.0,0.469414,0.111193,0.358221,0.000000
2,deep_control,2026,0.858307,2.222228,0.080096,0.273857,0.069004,1.0,1.0,0.480948,0.188272,0.292677,0.000000
3,deep_glcm_additive,42,0.858021,2.221488,0.080207,0.274235,0.068215,1.0,1.0,0.485928,0.160667,0.325261,0.002095
4,deep_glcm_additive,1337,0.855273,2.214372,0.083423,0.285232,0.070951,1.0,1.0,0.472094,0.183568,0.288526,0.002355
5,deep_glcm_additive,2026,0.855194,2.214169,0.082011,0.280404,0.070164,1.0,1.0,0.473427,0.183086,0.290342,0.001277


{
  "promote_to_locked_holdout": false,
  "criteria": {
    "qwk_gain_at_least_0_01": false,
    "macro_f1_gain_at_least_0_01": false,
    "macro_ap_no_material_regression": true,
    "grade1_recall_drop_within_0_03": true,
    "severe_error_does_not_increase": true,
    "ece_increase_within_0_01": true,
    "cam_gate_drop_less_than_0_05": true,
    "joint_occlusion_contrast_preserved": true,
    "qwk_direction_consistent": false,
    "macro_f1_direction_consistent": false,
    "glcm_branch_not_dominant": true
  }
}
Complete run artifacts: /content/drive/MyDrive/Models/densenet121_glcm_fusion/2026-07-31_00-49-00_429979_UTC
